In [33]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

### 1. Load Dataset & Initial Inspection

Load the Sales_transactions_2022_2025.csv dataset and inspect its dimensions and data types before cleaning.

In [34]:
df = pd.read_csv('Sales_transactions_2022_2025.csv')
df.columns = df.columns.str.strip()
print(list(zip(df.columns, df.dtypes)))

[('Transaction_ID', <StringDtype(storage='python', na_value=nan)>), ('Order_ID', <StringDtype(storage='python', na_value=nan)>), ('Customer_ID', <StringDtype(storage='python', na_value=nan)>), ('Customer_Name', <StringDtype(storage='python', na_value=nan)>), ('Customer_Age', <StringDtype(storage='python', na_value=nan)>), ('Customer_Gender', <StringDtype(storage='python', na_value=nan)>), ('Customer_Segment', <StringDtype(storage='python', na_value=nan)>), ('Order_Date', <StringDtype(storage='python', na_value=nan)>), ('Order_Time', <StringDtype(storage='python', na_value=nan)>), ('Sales_Channel', <StringDtype(storage='python', na_value=nan)>), ('Store_ID', <StringDtype(storage='python', na_value=nan)>), ('Store_Name', <StringDtype(storage='python', na_value=nan)>), ('Country', <StringDtype(storage='python', na_value=nan)>), ('Region', <StringDtype(storage='python', na_value=nan)>), ('City', <StringDtype(storage='python', na_value=nan)>), ('Product_ID', <StringDtype(storage='python', n

### 2. Feature Selection & Data Leakage Prevention

Remove unique IDs (which cause overfitting) and post-purchase fields like Return_Reason and Order_Status so the model only learns from information available at the time of purchase.

In [35]:
target = 'Return_Flag'
drop_cols = [
    'Transaction_ID', 'Order_ID', 'Customer_ID', 'Customer_Name',
    'Store_ID', 'Product_ID', 'Sales_Representative',
    'Return_Reason', 'Order_Status', 'Customer_Rating', 'Sales_Amount',
    'Order_Date', 'Order_Time'
]

X = df.drop(columns=[c for c in drop_cols + [target] if c in df.columns])
y = df[target]


### 3. Split the Data for Traning, Validating, and Testing (80% / 10% / 10%)

Split the dataset before any imputation or scaling to prevent data leakage, using stratification to preserve the return-to-non-return ratio across all three sets.

In [36]:
# Step 1: Split into 80% Train and 20% Temp (Validation + Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Step 2: Split the 20% Temp set equally into 10% Validation and 10% Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train shape: {X_train.shape}")
print(f"Validation shape: {X_val.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (14436, 22)
Validation shape: (1804, 22)
Test shape: (1805, 22)


### 4. Numeric Type Casting & Target Encoding

Convert numeric fields stored as strings into true numeric types and map Return_Flag (True/False) into binary labels.

In [37]:
to_numeric = [
    'Customer_Age', 'Quantity', 'Unit_Price', 'Discount_Percentage',
    'Cost_Amount', 'Profit', 'Delivery_Days', 'Inventory_Level', 'Order_Year'
]

# Keep only the ones still present in X_train
num_cols = [col for col in to_numeric if col in X_train.columns]

# Convert strings to actual numbers
for col in num_cols:
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
    X_val[col] = pd.to_numeric(X_val[col], errors='coerce')
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce')

y_train = y_train.astype(str).str.strip().str.lower().map({'true': 1, 'false': 0, '1': 1, '0': 0})
y_val   = y_val.astype(str).str.strip().str.lower().map({'true': 1, 'false': 0, '1': 1, '0': 0})
y_test  = y_test.astype(str).str.strip().str.lower().map({'true': 1, 'false': 0, '1': 1, '0': 0})

print("Target class counts (y_train):")
print(y_train.value_counts(dropna=False))
print("\nTarget percentages:")
print(y_train.value_counts(normalize=True) * 100)

Target class counts (y_train):
Return_Flag
NaN    14436
Name: count, dtype: int64

Target percentages:
Series([], Name: proportion, dtype: float64)


### 5. Imputation, One-Hot Encoding & Feature Scaling

Fit a ColumnTransformer strictly on the training data to impute missing values, one-hot encode categorical text features, and standardize numeric features (required for Logistic Regression and our MLP neural network).

In [38]:
# Dynamically grab numeric vs. categorical columns from X_train
num_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist()

print("Numeric features:", num_features)
print("Categorical features:", cat_features)

# Numeric pipeline: Median imputation + Standard Scaling
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline: Constant imputation + One-Hot Encoding
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine both into a single ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_features),
        ('cat', cat_pipeline, cat_features)
    ]
)

# Fit ONLY on X_train, then transform all three splits!
X_train_ready = preprocessor.fit_transform(X_train)
X_val_ready = preprocessor.transform(X_val)
X_test_ready = preprocessor.transform(X_test)

print(f"Final Train matrix: {X_train_ready.shape}")
print(f"Final Val matrix:   {X_val_ready.shape}")
print(f"Final Test matrix:  {X_test_ready.shape}")

display(X_train[num_cols].describe().round(2))

Numeric features: ['Customer_Age', 'Quantity', 'Unit_Price', 'Discount_Percentage', 'Cost_Amount', 'Profit', 'Delivery_Days', 'Inventory_Level', 'Order_Year']
Categorical features: ['Customer_Gender', 'Customer_Segment', 'Sales_Channel', 'Store_Name', 'Country', 'Region', 'City', 'Product_Name', 'Product_Category', 'Product_Subcategory', 'Payment_Method', 'Shipping_Method', 'Promotion_Code']
Final Train matrix: (14436, 151)
Final Val matrix:   (1804, 151)
Final Test matrix:  (1805, 151)


,Customer_Age,Quantity,Unit_Price,Discount_Percentage,Cost_Amount,Profit,Delivery_Days,Inventory_Level,Order_Year
count,14266.00,14436.00,14436.00,14436.00,14436.00,14436.00,14309.00,14268.00,14436.00
mean,38.91,2.12,232.27,6.23,365.02,91.83,2.27,70.21,2023.57
std,10.66,1.78,293.34,6.87,704.68,170.35,2.33,22.05,1.12
min,15.00,-3.00,21.57,0.00,12.17,-299.31,0.00,0.00,2022.00
25%,32.00,1.00,53.22,0.00,56.65,19.52,0.00,55.00,2023.00
50%,39.00,2.00,99.91,5.00,122.38,40.44,1.00,70.00,2024.00
75%,46.00,3.00,284.76,10.00,373.70,92.97,4.00,85.00,2025.00
max,99.00,11.00,1425.09,110.00,10791.00,3512.08,10.00,157.00,2025.00
